In [1]:
# Loading the data
import pandas as pd

imdb_dataset = pd.read_csv("hf://datasets/scikit-learn/imdb/IMDB Dataset.csv")
imdb_dataset.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [2]:
# Calculate the number of words in each review and find the average
average_words = imdb_dataset['review'].str.split().str.len().mean()
print(f"Average number of words per review: {average_words:.2f}")

Average number of words per review: 231.16


<h1 align='center'>Text Cleaning</h1>

In [3]:
# Lower casing
imdb_dataset['review'] = imdb_dataset['review'].apply(lambda x: x.lower())
print(imdb_dataset.head())

                                              review sentiment
0  one of the other reviewers has mentioned that ...  positive
1  a wonderful little production. <br /><br />the...  positive
2  i thought this was a wonderful way to spend ti...  positive
3  basically there's a family where a little boy ...  negative
4  petter mattei's "love in the time of money" is...  positive


In [6]:
# Removing a whitespace

from tqdm.notebook import tqdm

def remove_extra_space(text):
  text = text.strip()
  text = " ".join(text.split())
  return text

imdb_dataset['review'] = [remove_extra_space(review) for review in tqdm(imdb_dataset['review'], desc = "Cleaning extra spaces")]

Cleaning extra spaces:   0%|          | 0/50000 [00:00<?, ?it/s]

In [10]:
# Removing the HTML tags

from bs4 import BeautifulSoup

def remove_extra_space(text):
  soup = BeautifulSoup(text, "html.parser")
  return soup.get_text()

imdb_dataset['review'] = [remove_extra_space(review) for review in tqdm(imdb_dataset['review'], desc = "Cleaning HTML tags")]
print(imdb_dataset.head())

Cleaning HTML tags:   0%|          | 0/50000 [00:00<?, ?it/s]

                                              review sentiment
0  one of the other reviewers has mentioned that ...  positive
1  a wonderful little production. the filming tec...  positive
2  i thought this was a wonderful way to spend ti...  positive
3  basically there's a family where a little boy ...  negative
4  petter mattei's "love in the time of money" is...  positive


In [11]:
# Remove special characters

import re

def remove_special_characters(text):
  text = re.sub(r"[^a-zA-Z0-9\s]", "", text)
  return text

imdb_dataset['review'] = [remove_special_characters(review) for review in tqdm(imdb_dataset['review'], desc = "Cleaning special characters")]

Cleaning special characters:   0%|          | 0/50000 [00:00<?, ?it/s]

In [12]:
!pip install contractions

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 345.1/345.1 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.8/114.8 kB 8.4 MB/s eta 0:00:00


In [13]:
# Expanding words (don't -> do not)
import contractions

def expand_words(text):
  return contractions.fix(text)

imdb_dataset['review'] = [expand_words(review) for review in tqdm(imdb_dataset['review'], desc = "Expanding words")]
imdb_dataset.head()

Expanding words:   0%|          | 0/50000 [00:00<?, ?it/s]

,review,sentiment
0,one of the other reviewers has mentioned that ...,positive
1,a wonderful little production the filming tech...,positive
2,i thought this was a wonderful way to spend ti...,positive
3,basically there is a family where a little boy...,negative
4,petter matteis love in the time of money is a ...,positive


In [15]:
# Removing stop words
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
nltk.download('stopwords')
nltk.download('punkt_tab')

def remove_stop_words(text):
  stop_words = set(stopwords.words('english'))
  word_tokens = word_tokenize(text)
  filtered_words = [w for w in word_tokens if w not in stop_words]
  return " ".join(filtered_words)

imdb_dataset['review'] = [remove_stop_words(review) for review in tqdm(imdb_dataset['review'], desc = "Removing stop words")]
imdb_dataset.head()

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


Removing stop words:   0%|          | 0/50000 [00:00<?, ?it/s]

,review,sentiment
0,one reviewers mentioned watching 1 oz episode ...,positive
1,wonderful little production filming technique ...,positive
2,thought wonderful way spend time hot summer we...,positive
3,basically family little boy jake thinks zombie...,negative
4,petter matteis love time money visually stunni...,positive


In [16]:
# lemmatization

import nltk
nltk.download('wordnet')
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize

def lemmatize_text(text):
  lemmatizer = WordNetLemmatizer()
  word_tokens = word_tokenize(text)
  lemmatized_words = [lemmatizer.lemmatize(w) for w in word_tokens]
  return " ".join(lemmatized_words)

imdb_dataset['review'] = [lemmatize_text(review) for review in tqdm(imdb_dataset['review'], desc = "Lemmatizing")]
imdb_dataset.head()

[nltk_data] Downloading package wordnet to /root/nltk_data...


Lemmatizing:   0%|          | 0/50000 [00:00<?, ?it/s]

,review,sentiment
0,one reviewer mentioned watching 1 oz episode h...,positive
1,wonderful little production filming technique ...,positive
2,thought wonderful way spend time hot summer we...,positive
3,basically family little boy jake think zombie ...,negative
4,petter matteis love time money visually stunni...,positive


<h1>Training, Validation and Test</h1>

In [17]:
# labeling
imdb_dataset['sentiment'] = imdb_dataset['sentiment'].map({'positive': 1, 'negative': 0})
imdb_dataset.head()

,review,sentiment
0,one reviewer mentioned watching 1 oz episode h...,1
1,wonderful little production filming technique ...,1
2,thought wonderful way spend time hot summer we...,1
3,basically family little boy jake think zombie ...,0
4,petter matteis love time money visually stunni...,1


In [18]:
#Dividing the text

from sklearn.model_selection import train_test_split

train_df, temp_df = train_test_split(imdb_dataset, test_size=0.2, random_state=42)
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42)

print("Train set size: ", len(train_df))
print("Validation set size: ", len(val_df))
print("Test set size: ", len(test_df))

Train set size:  40000
Validation set size:  5000
Test set size:  5000


In [19]:
# Labels

train_df_labels = train_df['sentiment'].values
val_df_labels = val_df['sentiment'].values
test_df_labels = test_df['sentiment'].values

In [20]:
# Tokenizing
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf_vectorizer = TfidfVectorizer(max_features=1000)

train_df_tfidf = tfidf_vectorizer.fit_transform(train_df['review'])
val_df_tfidf = tfidf_vectorizer.transform(val_df['review'])
test_df_tfidf = tfidf_vectorizer.transform(test_df['review'])

<h1>Model Building and Training<h1>

In [23]:
import tensorflow as tf

model_tfidf = tf.keras.models.Sequential([
    tf.keras.layers.Dense(1, activation='sigmoid', input_shape=(train_df_tfidf.shape[1],))
])

model_tfidf.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model_tfidf.fit(train_df_tfidf, train_df_labels, epochs=5, batch_size=50, validation_data=(val_df_tfidf, val_df_labels))

Epoch 1/5
800/800 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.7855 - loss: 0.6307 - val_accuracy: 0.8336 - val_loss: 0.5746
Epoch 2/5
800/800 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.8359 - loss: 0.5372 - val_accuracy: 0.8382 - val_loss: 0.5028
Epoch 3/5
800/800 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - accuracy: 0.8432 - loss: 0.4793 - val_accuracy: 0.8436 - val_loss: 0.4567
Epoch 4/5
800/800 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.8483 - loss: 0.4405 - val_accuracy: 0.8480 - val_loss: 0.4251
Epoch 5/5
800/800 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.8527 - loss: 0.4131 - val_accuracy: 0.8476 - val_loss: 0.4021


<h1>TESTING<h1>

In [25]:
loss_tf_idf, accuracy_tf_idf = model_tfidf.evaluate(test_df_tfidf, test_df_labels)
print("Test accuracy in percentage:", accuracy_tf_idf * 100)

157/157 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8460 - loss: 0.4071
Test accuracy in percentage: 84.60000157356262
